In [6]:
import numpy as np
import pandas as pd 
import re

In [8]:
#chargement des données
dataset_gbm = pd.read_csv('Dataset_gbm.csv')
seq_dataset = pd.read_excel('whole_proteome_uniprot_id.xlsx')
display(dataset_gbm.head())
display(seq_dataset.head())

c:\Users\nicol\miniconda3\envs\gioblastoma\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,UniProt ID,Gene,Mutation,Class
0,Q5JS54,PSMG4,L112P,Driver
1,O14715,RGPD8,P1760A,Driver
2,Q6NW40,RGMB,L206F,Driver
3,P40306,PSB10,R58G,Driver
4,Q5VXH4,PRAM6,C166Y,Driver


,From,Entry,Entry Name,Protein names,Gene Names,Length,Sequence,Reviewed
0,Q9Y6K1,Q9Y6K1,DNM3A_HUMAN,DNA (cytosine-5)-methyltransferase 3A (Dnmt3a)...,DNMT3A,912,MPAMPSSGPGDTSSSAAEREEDRKDGEEQEEPRGKEERQEPSTTAR...,reviewed
1,O75874,O75874,IDHC_HUMAN,Isocitrate dehydrogenase [NADP] cytoplasmic (I...,IDH1 PICD,414,MSKKISGGSVVEMQGDEMTRIIWELIKEKLIFPYVELDLHSYDLGI...,reviewed
2,P48735,P48735,IDHP_HUMAN,"Isocitrate dehydrogenase [NADP], mitochondrial...",IDH2,452,MAGYLRVVRSLCRASGSRPAWAPAALTAPTSQEQPRRHYADKRIKV...,reviewed
3,P36888,P36888,FLT3_HUMAN,Receptor-type tyrosine-protein kinase FLT3 (EC...,FLT3 CD135 FLK2 STK1,993,MPALARDGGQLPLLVVFSAMIFGTITNQDLPVIKCVLINHKNNDSS...,reviewed
4,P01116,P01116,RASK_HUMAN,GTPase KRas (EC 3.6.5.2) (K-Ras 2) (Ki-Ras) (c...,KRAS KRAS2 RASK2,189,MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVI...,reviewed


In [9]:
def parse_mutation(mutation_str):
    """Parse la mutation, pour accessibilité facile à WT, position, mutant"""
    match = re.match(r"([A-Z])(\d+)([A-Z])", mutation_str)
    if match:
        wt, pos, mut = match.groups()
        return wt, int(pos), mut
    return None, None, None

# Appliquer la fonction de parsing à la colonne 'Mutation' et créer les nouvelles colonnes
dataset_gbm[['Wildtype', 'Position', 'Mutant']] = dataset_gbm['Mutation'].apply(lambda x: pd.Series(parse_mutation(x)))

# On effectue la jointure
df = pd.merge(
    dataset_gbm, 
    seq_dataset[['From', 'Sequence']], # On ne sélectionne que les colonnes utiles de seq_dataset pour ne pas alourdir dataset_gbm
    left_on='UniProt ID',      # La clé de jointure dans le 1er dataframe
    right_on='From',           # La clé de jointure dans le 2ème dataframe
    how='inner'                # 'inner' garde uniquement les correspondances communes (supprime les orphelins)
)

# On supprime la colonne 'From' qui fait doublon avec 'UniProt ID'
df = df.drop(columns=['From'])

# Affichage du résultat
print(df.head())
print(df.columns)
print(df.shape)

  UniProt ID   Gene Mutation   Class Wildtype  Position Mutant  \
0     Q5JS54  PSMG4    L112P  Driver        L       112      P   
1     O14715  RGPD8   P1760A  Driver        P      1760      A   
2     Q6NW40   RGMB    L206F  Driver        L       206      F   
3     P40306  PSB10     R58G  Driver        R        58      G   
4     Q5VXH4  PRAM6    C166Y  Driver        C       166      Y   

                                            Sequence  
0  MEGLVVAAGGDVSLHNFSARLWEQLVHFHVMRLTDSLFLWVGATPH...  
1  MRRSKADVERYVASVLGLTPSPRQKSMKGFYFAKLYYEAKEYDLAK...  
2  MGLRAAPSSAAAAAAEVEQRRSPGLCPPPLELLLLLLFSLGLLHAG...  
3  MLKPALEPRGGFSFENCQRNASLERVLPGLKVPHARKTGTTIAGLV...  
4  MSIRTPPRLLELAGRSLLRDQALAMSTLEELPTELFPPLFMEAFSR...  
Index(['UniProt ID', 'Gene', 'Mutation', 'Class', 'Wildtype', 'Position',
       'Mutant', 'Sequence'],
      dtype='object')
(18062, 8)


In [20]:
def extract_motifs(row):
    """Extrait les 7 types de motifs autour du site de mutation."""
    seq = str(row['Sequence'])
    mutant = str(row['Mutant'])
    
    # IMPORTANT : On suppose que la colonne 'pos' est basée sur 1 (standard biologique).
    # On la convertit en index Python (basé sur 0).
    pos = int(row['Position']) - 1 
    seq_len = len(seq)
    m = {}
    
    # Voisinage direct (XMX, XM, MX)
    m['XMX'] = seq[pos-1] + mutant + seq[pos+1] if 0 < pos < seq_len-1 else np.nan
    m['XM']  = seq[pos-1] + mutant if 0 < pos else np.nan
    m['MX']  = mutant + seq[pos+1] if pos < seq_len-1 else np.nan
    
    # Voisinage avec 1 Gap (X*M, M*X)
    m['X*M'] = seq[pos-2] + "*" + mutant if 1 < pos else np.nan
    m['M*X'] = mutant + "*" + seq[pos+2] if pos < seq_len-2 else np.nan
    
    # Voisinage avec 2 Gaps (X**M, M**X)
    m['X**M'] = seq[pos-3] + "**"+ mutant if 2 < pos else np.nan
    m['M**X'] = mutant + "**" + seq[pos+3] if pos < seq_len-3 else np.nan
    
    return pd.Series(m)

# --- NETTOYAGE DES SÉQUENCES VIDES ---
print(f"Nombre de lignes avant nettoyage : {len(df)}")

# 1. On force toutes les cases vides (ou qui ne contiennent que des espaces) à devenir de vrais 'NaN'
df['Sequence'] = df['Sequence'].replace(r'^\s*$', np.nan, regex=True)

# 2. On supprime toutes les lignes où la séquence est un 'NaN'
df = df.dropna(subset=['Sequence'])
#---------------------------------------------------------

# --- FILTRAGE DES POSITIONS HORS LIMITES ---
# 1. On s'assure que la colonne 'Position' est bien lue comme des nombres
# (errors='coerce' transformera toute valeur bizarre en 'NaN')
df['Position'] = pd.to_numeric(df['Position'], errors='coerce')

# 2. On crée un masque pour garder uniquement :
# - Les positions supérieures à 0 (biologiquement valide)
# - Les positions inférieures ou égales à la longueur de la séquence
df = df[(df['Position'] > 0) & (df['Position'] <= df['Sequence'].str.len())]

print(f"Nombre de lignes après filtrage : {len(df)}")
# ---------------------------------------

print("Extraction des motifs...")
# Application de la fonction d'extraction et concaténation avec le dataset principal
motifs_df = df.apply(extract_motifs, axis=1)
motifs_df['Class'] = df['Class']
print("Motifs extraits")


Nombre de lignes avant nettoyage : 18057
Nombre de lignes après filtrage : 18057
Extraction des motifs...
Motifs extraits


In [43]:
# --- PARTIE 1 : CALCULS GLOBAUX (Inchangé) ---
Nd = (motifs_df['Class'] == 'Driver').sum()
Np = (motifs_df['Class'] == 'Passenger').sum()

# --- PARTIE 2 : RESTRUCTURATION (Modifiée pour garder le type de motif) ---
colonnes_motifs = ['XMX', 'XM', 'MX', 'X*M', 'M*X', 'X**M', 'M**X']

# MODIF : On ajoute var_name='Type_Motif' pour savoir de quelle colonne vient le motif
df_melted = motifs_df.melt(
    id_vars=['Class'], 
    value_vars=colonnes_motifs, 
    var_name='Type_Motif', 
    value_name='Motif'
)

# MODIF : On groupe par [Type_Motif, Motif] au lieu de juste Motif
df_results = pd.crosstab(
    [df_melted['Type_Motif'], df_melted['Motif']], 
    df_melted['Class']
).reset_index()

# Sécurisation et renommage (Adapté à tes nouveaux noms)
if 'Driver' not in df_results.columns: df_results['Driver'] = 0
if 'Passenger' not in df_results.columns: df_results['Passenger'] = 0

df_results = df_results.rename(columns={'Driver': 'count_x', 'Passenger': 'count_y'})

# --- PARTIE 3 : CALCUL DE L'ODDS RATIO (Logique identique, noms de colonnes mis à jour) ---
df_results['odd_ratio'] = 0.0
mask_ndp_0 = (df_results['count_x'] == 0)
mask_nnp_0 = (df_results['count_y'] == 0)
mask_normal = ~(mask_ndp_0 | mask_nnp_0)

df_results.loc[mask_ndp_0, 'odd_ratio'] = 0
df_results.loc[mask_nnp_0 & ~mask_ndp_0, 'odd_ratio'] = 50
df_results.loc[mask_normal, 'odd_ratio'] = (
    (df_results.loc[mask_normal, 'count_x'] / Nd) / 
    (df_results.loc[mask_normal, 'count_y'] / Np)
)

# --- PARTIE 4 : AJOUT DES COLONNES STATIQUES ---
df_results['class_x'] = 1
df_results['class_y'] = 0

# --- NOUVEAU : Dictionnaire de correspondance des noms ---
noms_onglets = {
    'XMX': 'tri',
    'XM': 'nM',
    'MX': 'Mc',
    'X*M': 'n_M',
    'M*X': 'M_c',
    'X**M': 'n__M',
    'M**X': 'M__c'
}

# --- PARTIE 5 : CRÉATION DU FICHIER EXCEL ---
with pd.ExcelWriter('GBM_odds_ratios_computed.xlsx') as writer:
    for cat in colonnes_motifs:
        # On filtre pour l'onglet actuel
        df_sheet = df_results[df_results['Type_Motif'] == cat].copy()
        
        # On sélectionne les colonnes dans l'ordre demandé
        df_sheet = df_sheet[['Motif', 'class_x', 'count_x', 'class_y', 'count_y', 'odd_ratio']]
        
        # On trie par importance
        df_sheet = df_sheet.sort_values(by='odd_ratio', ascending=False)
        
        # On récupère le nouveau nom stylisé grâce au dictionnaire ('tri', 'nM', etc.)
        nouveau_nom = noms_onglets[cat]
        
        # --- NOUVEAUTÉ : On renomme dynamiquement la colonne 'Motif' ---
        df_sheet = df_sheet.rename(columns={'Motif': nouveau_nom})
        
        # On écrit dans l'onglet avec le nouveau nom
        df_sheet.to_excel(writer, sheet_name=nouveau_nom, index=False)

print("Fichier Excel généré avec 7 onglets (caractères spéciaux nettoyés) !")

Fichier Excel généré avec 7 onglets (caractères spéciaux nettoyés) !
